# Bixi + Metro: from open feeds to a map

Real-time bike and dock counts are not stored here: they change every few seconds, so the page reads them live from BIXI's `station_status` feed and joins them to these stations by `station_id`.

Install GeoPandas

In [3]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

AttributeError: partially initialized module 'pandas' from '/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

## Step 1. BIXI stations 

BIXI publishes its stations as an open GBFS feed: https://gbfs.velobixi.com/gbfs/en/station_information.json (saved in `data/bixi_station_information.json`). Each station already has `lat` and `lon` in degrees. The box around Greater Montréal drops 33 stations: 4 test stations at 0, 0, 25 in Sherbrooke and 4 in Chambly. That leaves 1,088.

In [ ]:
# Saved from https://gbfs.velobixi.com/gbfs/en/station_information.json
with open("../data/bixi_station_information.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f)["data"]["stations"])
n_all = len(df)
print("all stations in the feed:", n_all)


in_box = df["lat"].between(45.3, 45.8) & df["lon"].between(-74.1, -73.3)
print("dropped, outside Montréal:", (~in_box).sum())
df = df[in_box]

before = len(df)
df = df.drop_duplicates("station_id")
print("dropped, duplicate ids:", before - len(df))
print("kept:", len(df), "of", n_all)

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
gdf = gdf[["station_id", "name", "capacity", "geometry"]]


all stations in the feed: 1121
dropped, outside Montréal: 33
dropped, duplicate ids: 0
kept: 1088 of 1121


## Step 2. Cross-listing BIXI stations with metro stations

In [ ]:
WALK_M = 300   

m = pd.read_csv("../data/stm_metro_stations.csv")
metro = gpd.GeoDataFrame(m, geometry=gpd.points_from_xy(m["lon"], m["lat"]), crs="EPSG:4326")
print("metro stations:", len(metro))

bixi_m = gdf[["station_id", "name", "capacity", "geometry"]].to_crs("EPSG:32188")
metro_m = metro[["metro_id", "metro_name", "lines", "geometry"]].to_crs("EPSG:32188")

bixi_m = gpd.sjoin_nearest(bixi_m, metro_m, distance_col="metro_dist_m").drop(columns="index_right")
bixi_m = bixi_m.drop_duplicates("station_id")          # a tie between two metros would give two rows
bixi_m["metro_dist_m"] = bixi_m["metro_dist_m"].round().astype(int)
bixi_m["near_metro"] = bixi_m["metro_dist_m"] <= WALK_M
print(bixi_m["near_metro"].sum(), "of", len(bixi_m), f"BIXI stations are within {WALK_M} m of a metro station")

pairs = gpd.sjoin(metro_m, bixi_m[["station_id", "capacity", "geometry"]], predicate="dwithin", distance=WALK_M)
counts = pairs.groupby("metro_id").agg(bixi_nearby=("station_id", "size"), docks_nearby=("capacity", "sum"))
metro_m = metro_m.merge(counts, on="metro_id", how="left")
metro_m[["bixi_nearby", "docks_nearby"]] = metro_m[["bixi_nearby", "docks_nearby"]].fillna(0).astype(int)

bixi_m.to_crs("EPSG:4326").to_file("../data/bixi_stations.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
metro_m.to_crs("EPSG:4326").to_file("../data/metro_stations.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
print(round(os.path.getsize("../data/bixi_stations.geojson") / 1_000_000, 2), "MB and",
      round(os.path.getsize("../data/metro_stations.geojson") / 1_000_000, 3), "MB")

metro stations: 68
209 of 1088 BIXI stations are within 300 m of a metro station
0.33 MB and 0.015 MB


In [ ]:
print("Metro stations with no BIXI station within", WALK_M, "m:",
      metro_m.loc[metro_m["bixi_nearby"] == 0, "metro_name"].tolist())
metro_m[["metro_name", "lines", "bixi_nearby", "docks_nearby"]].sort_values("docks_nearby").reset_index(drop=True)

Metro stations with no BIXI station within 300 m: ['Langelier']


,metro_name,lines,bixi_nearby,docks_nearby
0,Langelier,Green,0,0
1,Henri-Bourassa,Orange,1,16
2,Du Collège,Orange,1,17
3,Place-Saint-Henri,Orange,1,18
4,Jolicoeur,Green,1,18
...,...,...,...,...
63,Square-Victoria–OACI,Orange,9,231
64,Bonaventure,Orange,7,247
65,Berri-UQAM,Green / Orange / Yellow,11,297
66,Peel,Green,14,483


In [ ]:
near = bixi_m[bixi_m["near_metro"]]
near[["name", "capacity", "metro_name", "lines", "metro_dist_m"]].sort_values("metro_dist_m").reset_index(drop=True)

,name,capacity,metro_name,lines,metro_dist_m
0,Métro Beaudry (Beaudry / Ste-Catherine),23,Beaudry,Green,10
1,Métro Lucien-L'Allier (Argyle),21,Lucien-L'Allier,Orange,12
2,Métro de la Concorde (Léo-Lacombe),19,De la Concorde,Orange,13
3,Métro De la Savane (Bougainville / de Sorel),19,De la Savane,Orange,16
4,Métro Lionel-Groulx (St-Jacques / Atwater),33,Lionel-Groulx,Green / Orange,16
...,...,...,...,...,...
204,du Square-St-Louis / Laval,35,Sherbrooke,Orange,297
205,Laval / du Mont-Royal,23,Mont-Royal,Orange,299
206,Hélène-Baillargeon / St-Denis,23,Rosemont,Orange,299
207,Crescent / de Maisonneuve,23,Guy-Concordia,Green,299


## Step 3. Check the files

In [ ]:
FILE = "../data/bixi_stations.geojson"
WEST, EAST, SOUTH, NORTH = -74.1, -73.3, 45.3, 45.8
EXPECTED = len(gdf)
POPUP_FIELDS = ["station_id", "name", "capacity", "metro_name", "lines", "metro_dist_m", "near_metro"]

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
All checks passed


In [ ]:
# And the metro file: 68 stations, in degrees, with the counts the map shows
back = gpd.read_file("../data/metro_stations.geojson")
ok = len(back) == 68 and back.crs.to_epsg() == 4326 and all(f in back.columns for f in ["metro_name", "lines", "bixi_nearby", "docks_nearby"])
print("Metro file OK" if ok else "Metro file: check the count, CRS and columns")

Metro file OK
